# Phase 1 EDA — CICIoT2023 dataset track

This notebook starts the modern IoT branch of the intrusion-detection project. It does **not** replace NSL-KDD; it diversifies the portfolio by adding a recent, larger, IoT-specific dataset.

Source verified from UNB/CIC: <https://www.unb.ca/cic/datasets/iotdataset-2023.html>


## 1. Why this dataset

CICIoT2023 is a good next dataset because the official UNB/CIC page describes:

- 105 IoT devices in the topology.
- 33 attacks.
- 7 attack categories: DDoS, DoS, Recon, Web-based, Brute Force, Spoofing, and Mirai.
- CSV flow features, PCAPs, an example notebook, and supplementary tooling.

That makes it much more useful for modern IoT intrusion detection than simply adding another old benchmark. The Phase-1 goal is not model training yet; it is to understand labels, imbalance, feature quality, and practical scale.


In [1]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "src").exists():
    REPO = CWD
elif (CWD.parent / "src").exists():
    REPO = CWD.parent
else:
    raise RuntimeError(f"Could not locate repo root from {CWD}")
SRC = REPO / "src"
sys.path.insert(0, str(SRC))

os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".matplotlib-cache"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import ciciot2023 as C

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
print(f"repo: {REPO}")
print(f"CICIoT2023 CSV dir: {C.CSV_DIR}")


repo: /Users/guneyaliunal/Desktop/don't delete/dazed&confused/nsl-kdd-ids
CICIoT2023 CSV dir: /Users/guneyaliunal/Desktop/don't delete/dazed&confused/nsl-kdd-ids/data/ciciot2023/CSV


In [2]:
source_card = pd.DataFrame([
    {"field": "dataset", "value": "CICIoT2023"},
    {"field": "provider", "value": "Canadian Institute for Cybersecurity, University of New Brunswick"},
    {"field": "source", "value": C.DATASET_PAGE},
    {"field": "topology", "value": "105 IoT devices"},
    {"field": "attack coverage", "value": "33 attacks in 7 attack categories"},
    {"field": "families", "value": ", ".join(C.CATEGORY_ORDER)},
    {"field": "local CSV path", "value": str(C.CSV_DIR.relative_to(REPO))},
])
display(source_card)


,field,value
0,dataset,CICIoT2023
1,provider,"Canadian Institute for Cybersecurity, Universi..."
2,source,https://www.unb.ca/cic/datasets/iotdataset-202...
3,topology,105 IoT devices
4,attack coverage,33 attacks in 7 attack categories
5,families,"Benign, DDoS, DoS, Recon, Web-based, Brute For..."
6,local CSV path,data/ciciot2023/CSV


## 2. Local data discovery

Expected local layout after download/extract:

```text
data/ciciot2023/CSV/*.csv
```

The project intentionally does not commit those CSVs. They are bulk data and are ignored by `.gitignore`; only `data/ciciot2023/SOURCE.md` is committed.


In [3]:
csv_files = C.discover_csv_files()
if not csv_files:
    display(Markdown(
        "**No local CICIoT2023 CSV files found yet.** Download the CSV release from "
        f"[{C.DATASET_PAGE}]({C.DATASET_PAGE}) and place the extracted CSV files under "
        "`data/ciciot2023/CSV/`. The rest of this notebook is written to run once those files exist."
    ))
else:
    files_df = pd.DataFrame({"file": [str(p.relative_to(REPO)) for p in csv_files], "bytes": [p.stat().st_size for p in csv_files]})
    display(files_df)


**No local CICIoT2023 CSV files found yet.** Download the CSV release from [https://www.unb.ca/cic/datasets/iotdataset-2023.html](https://www.unb.ca/cic/datasets/iotdataset-2023.html) and place the extracted CSV files under `data/ciciot2023/CSV/`. The rest of this notebook is written to run once those files exist.

## 3. Label mapping smoke test

Before touching the full data, verify that our coarse category mapper handles the official-style fine labels. This is the equivalent of the NSL-KDD `attack_family` mapping guardrail.


In [4]:
label_examples = [
    "BenignTraffic",
    "DDoS-UDP_Flood",
    "DDoS-SlowLoris",
    "DoS-SYN_Flood",
    "Recon-PortScan",
    "VulnerabilityScan",
    "SqlInjection",
    "CommandInjection",
    "DictionaryBruteForce",
    "DNS_Spoofing",
    "Mirai-greeth_flood",
]
label_map_df = pd.DataFrame({
    "fine_label": label_examples,
    "category": [C.attack_category(x) for x in label_examples],
})
display(label_map_df)


,fine_label,category
0,BenignTraffic,Benign
1,DDoS-UDP_Flood,DDoS
2,DDoS-SlowLoris,DDoS
3,DoS-SYN_Flood,DoS
4,Recon-PortScan,Recon
5,VulnerabilityScan,Recon
6,SqlInjection,Web-based
7,CommandInjection,Web-based
8,DictionaryBruteForce,Brute Force
9,DNS_Spoofing,Spoofing


## 4. Phase-1 sample EDA

This section uses a bounded sample per CSV file. That is deliberate: CICIoT2023 is large enough that Phase 1 should be chunk/sample-aware before full modelling.


In [5]:
if csv_files:
    sample = C.load_csv_sample(max_rows_per_file=25_000)
    summary = C.phase1_summary(sample)
    print(f"sample rows: {summary['rows']:,}")
    print(f"feature columns: {summary['n_features']}")
    print(f"fine labels in sample: {summary['n_fine_labels']}")
    print(f"coarse categories in sample: {summary['n_categories']}")
    display(summary["category_counts"].rename("rows").to_frame())
    display(summary["label_counts"].rename("rows").head(30).to_frame())
else:
    sample = None
    display(Markdown("Skipping sample EDA because the local CSV files are not present yet."))


Skipping sample EDA because the local CSV files are not present yet.

In [6]:
if sample is not None:
    category_counts = sample["attack_category"].value_counts().reindex(C.CATEGORY_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(9, 4))
    category_counts.plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title("CICIoT2023 sample: coarse category balance")
    ax.set_ylabel("rows")
    ax.set_xlabel("category")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No plot until CSV files are present.")


No plot until CSV files are present.


In [7]:
if sample is not None:
    features = C.feature_columns(sample)
    numeric = sample[features].select_dtypes(include=[np.number])
    missing = sample[features].isna().mean().sort_values(ascending=False)
    constant_cols = [c for c in numeric.columns if numeric[c].nunique(dropna=False) <= 1]
    print(f"numeric feature columns: {numeric.shape[1]}")
    print(f"constant numeric columns: {len(constant_cols)}")
    if constant_cols:
        print(constant_cols[:30])
    display(missing.head(20).rename("missing_rate").to_frame())
    display(numeric.describe().T[["mean", "std", "min", "50%", "max"]].head(30))
else:
    print("No feature-quality audit until CSV files are present.")


No feature-quality audit until CSV files are present.


## 5. Tests for this Phase-1 track

The tests use synthetic mini CSVs. They do not require the full CICIoT2023 download, so they can run on every machine.


In [8]:
def run_cmd(cmd: list[str], timeout: int = 120) -> subprocess.CompletedProcess:
    print("$", " ".join(cmd))
    out = subprocess.run(
        cmd,
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(out.stdout[-4000:])
    print("exit:", out.returncode)
    return out

check = run_cmd([str(REPO / ".venv" / "bin" / "python"), "-m", "pytest", "tests/test_ciciot2023.py", "-q"])
assert check.returncode == 0


$ /Users/guneyaliunal/Desktop/don't delete/dazed&confused/nsl-kdd-ids/.venv/bin/python -m pytest tests/test_ciciot2023.py -q


...............                                                          [100%]

exit: 0


## 6. Next modelling bridge

Once the Phase-1 EDA is clean, reuse the NSL-KDD audit lessons:

1. Start with Dummy, Logistic Regression, balanced Logistic Regression, RandomForest, LightGBM, and MLP.
2. Use binary labels first: `Benign` vs attack.
3. Use 8-way multiclass second: `Benign` + 7 attack categories.
4. Lead with macro-F1 and per-class recall, not accuracy.
5. Keep all tuning on train/validation only.
6. Consider a fine-label holdout validation protocol to mimic novel attacks.

The core project story becomes stronger: NSL-KDD teaches old official-split generalization; CICIoT2023 tests whether the same workflow survives modern IoT scale and richer attack taxonomy.
